In [ ]:
import pandas as pd

In [ ]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt

# Directory containing the CSVs
trip_dir = "./bikeshare_data/tracts_with_trips"
# Directory to save the histogram images
output_dir = "./histograms"
os.makedirs(output_dir, exist_ok=True)

# Process each city's tracts file
for path in glob.glob(os.path.join(trip_dir, "*_tracts_with_trips.csv")):
    # Extract city name from filename
    city = os.path.basename(path).split("_")[0]
    # Read the data
    df = pd.read_csv(path, dtype={"GEOID": str})
    # Compute density (trips per day per unit land area)
    df["trip_density"] = (df["trips_per_day"]/ 2) / (df["ALAND"] / 2_589_988.11)
    
    # Plot histogram
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(df["trip_density"].dropna(), bins=50, edgecolor="black")
    ax.set_xlabel("Trip Density (trips per day per sq mi)")
    ax.set_ylabel("Number of Census Tracts")
    ax.set_title(f"Distribution of Bike-trip Density for {city.capitalize()} Bike Share System")
    plt.tight_layout()
    
    # Save the figure
    out_path = os.path.join(output_dir, f"{city}_trip_density_histogram.png")
    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    
    print(f"Saved histogram for {city} to {out_path}")


In [ ]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt

# Directory containing the CSVs
trip_dir = "./bikeshare_data/tracts_with_trips"

# Choose four specific files to include
selected_files = [
    "boston_tracts_with_trips.csv",
    "chicago_tracts_with_trips.csv",
    "portland_tracts_with_trips.csv",
    "pittsburgh_tracts_with_trips.csv"
]

# Create 2x2 subplot layout
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()  # Flatten for easy iteration
fig.suptitle("Distributions of Bike-Trip Density Across Four U.S. Cities", fontsize=16)

# Now plot into axes[0,0], axes[0,1], etc.


for i, filename in enumerate(selected_files):
    path = os.path.join(trip_dir, filename)
    city = filename.split("_")[0].capitalize()
    
    df = pd.read_csv(path, dtype={"GEOID": str})
    df["trip_density"] = (df["trips_per_day"] / 2) / (df["ALAND"] / 2_589_988.11)  # trips/day per square mile

    ax = axes[i]
    ax.hist(df["trip_density"].dropna(), bins=50, edgecolor="black")
    ax.set_title(f"{city}")
    ax.set_xlabel("Trip Density (trips/day per sq mi)")
    ax.set_ylabel("Census Tracts")
    

# Adjust layout and save
plt.tight_layout()
plt.savefig("./histograms/four_city_trip_density_grid.png", dpi=150)
plt.show()


In [ ]:
import os, glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ------------------------------------------------------------------
# 1)  Read the empirical trip‑density vectors into a dict
# ------------------------------------------------------------------
trip_dir   = "./bikeshare_data/tracts_with_trips"
empirical  = {}                      # {city: 1‑D ndarray of densities}

for path in glob.glob(os.path.join(trip_dir, "*_tracts_with_trips.csv")):
    city = os.path.basename(path).split("_")[0].lower()
    df   = pd.read_csv(path, dtype={"GEOID": str})
    empirical[city] = (df["trips_per_day"] / df["ALAND"]).values

print("Loaded empirical cities:", list(empirical.keys()))

# ------------------------------------------------------------------
# 2)  Get your STL **predicted** raw density (z → raw)
#     – Here I assume you still have 'predicted_density_z'
#       and you've already added the raw density column.
# ------------------------------------------------------------------
# If you *only* have z‑scores, back‑transform:  raw = z⋅σ + μ
# mu = stl_gdf['predicted_density'].mean()
# sigma = stl_gdf['predicted_density'].std()
# raw_pred = stl_gdf['predicted_density_z'] * sigma + mu
#raw_pred = stl_gdf['predicted_density']      # already in raw units

# ------------------------------------------------------------------
# 3)  Choose a few representative empirical cities to overlay
# ------------------------------------------------------------------
repr_cities = ['philly', 'sf', 'dmv']   # pick any three that illustrate variety
colors      = ['gray',  'darkgray', 'lightgray']   # faint greys for overlay

# ------------------------------------------------------------------
# 4)  Plot overlays with log‑x axis
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 4))

# STL prediction histogram
# ax.hist(
#     raw_pred, bins=50, density=True, alpha=0.7, label="STL prediction",
#     edgecolor="black"
# )

# overlay empirical curves
for city, color in zip(repr_cities, colors):
    ax.hist(
        empirical[city], bins=50, density=True,
        histtype="stepfilled", alpha=0.3, color=color, label=city.upper()
    )

ax.set_xscale("log")
ax.set_xlabel("Trips / day / land‑area (raw density, log‑scale)")
ax.set_ylabel("Density (share of tracts)")
ax.set_title("Predicted STL Density vs. Empirical City Distributions")
ax.legend(frameon=True)
plt.tight_layout()

# save or show
plt.savefig("overlay_density_STL_vs_empirical.png", dpi=150)
plt.show()


In [ ]:
import os, glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ----------------------------------------------------------
# 1)  Load every empirical city’s raw density vector
# ----------------------------------------------------------
trip_dir   = "./bikeshare_data/tracts_with_trips"
empirical  = {}                             # {city: ndarray}

for path in glob.glob(os.path.join(trip_dir, "*_tracts_with_trips.csv")):
    city = os.path.basename(path).split("_")[0].lower()
    df   = pd.read_csv(path, dtype={"GEOID": str})
    empirical[city] = (df["trips_per_day"] / df["ALAND"]).values

# ----------------------------------------------------------
# 2)  Pick representative systems to overlay
# ----------------------------------------------------------
repr_cities = ['philly', 'sf', 'dmv']   # edit this list as you like
colors      = ['tab:blue', 'tab:orange', 'tab:green']

# ----------------------------------------------------------
# 3)  Common bin edges (0 → 99th‑pct of all shown curves)
# ----------------------------------------------------------
vals_concat = np.concatenate([empirical[c] for c in repr_cities])
x_max       = np.quantile(vals_concat, 0.99)   # trim extreme outliers
edges       = np.linspace(0, x_max, 50+1)      # 50 bins

# ----------------------------------------------------------
# 4)  Overlay plot (raw scale, density normalised)
# ----------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 4))

for city, col in zip(repr_cities, colors):
    ax.hist(empirical[city],
            bins=edges, density=True,
            alpha=0.4, edgecolor='none',
            label=city.upper(), color=col)

ax.set_xlabel("Trips per Day per Land Area (raw density)")
ax.set_ylabel("Share of Census Tracts (density=True)")
ax.set_title("Empirical Bike‑share Density Distributions\n(three representative systems)")

# switch to log–x (recommended baseline)
ax.set_xscale('log')
#ax.set_yscale('log')

ax.legend(frameon=True)
plt.tight_layout()
plt.savefig("empirical_density_overlay.png", dpi=150)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def log_binned_hist(ax, data, bins=40, label=None, color=None, alpha=0.5):
    """
    Plot a histogram with one 'zero' bin and log‑spaced bins for x>0.
    Parameters
    ----------
    ax     : matplotlib axis
    data   : 1‑D array‑like of densities (>=0)
    bins   : number of *log* bins for x>0
    label  : legend label
    color  : facecolor
    alpha  : alpha transparency
    """
    data = np.asarray(data)
    zeros = data == 0
    pos   = data[~zeros]

    if len(pos) == 0:                              # all zeros – unlikely
        ax.bar(0.25, zeros.sum(), width=0.5,
               align='center', color=color, alpha=alpha, label=label)
        return

    # Build log‑spaced bin edges for positive tail
    x_min = pos.min()
    x_max = np.quantile(pos, 0.99)                 # trim extreme right tail
    log_edges = np.geomspace(x_min, x_max, bins)
    # prepend 0 and a tiny epsilon so 0 falls into first bin
    edges = np.concatenate(([0, x_min*0.999], log_edges))

    # Histogram using the custom edges
    ax.hist(data, bins=edges, density=True,
            alpha=alpha, color=color, label=label, edgecolor='none')

    # Visual tweak: make the first (zero) bin visible as a separate bar
    #ax.set_xscale('symlog', linthresh= x_min/10)   # tiny linear region around 0
    ax.set_xscale('log')
    ax.set_yscale('log')   # tiny linear region around 0


# ------------------------------------------------------------------
# Example overlay of three empirical cities
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7,4))

colors = ['tab:blue', 'tab:orange', 'tab:green']
# for city, color in zip(['philly','sf','dmv'], colors):
#     log_binned_hist(ax, empirical[city], bins=40,
#                     label=city.upper(), color=color, alpha=0.35)

log_binned_hist(ax, empirical['sf'], bins=40,
                label=city.upper(), color=color, alpha=0.35)
ax.set_xlabel("Trips per Day per Land Area (log scale for x>0)")
ax.set_ylabel("Density (share of tracts)")
ax.set_title("Empirical Bike‑share Density Distributions (log‑binned)")
ax.legend()
plt.tight_layout()
plt.show()
